# 02 — Feature Engineering, Leakage Control & Train/Val/Test Split
### Lending Club Accepted Loans — Credit Risk (Default Prediction)

This notebook turns the raw file into three model-ready tables: `train.csv`,
`val.csv`, `test.csv`. Everything here is about getting the *foundation* right —
correct target, no leakage, sensible missing-value logic, a handful of engineered
ratios, and a split design that actually simulates deployment. Encoding
(target/one-hot) is deliberately **not** done here — it happens inside the modeling
`Pipeline` in notebook 03, fit only on training folds, so it can never leak
information from validation/test into training.

This extends the target definition, leakage-keyword screening, and domain-aware
imputation approach already used in the existing `preprocessing.ipynb` — the additions
here are the loan-maturity filter, the time-based (out-of-time) split, engineered ratio
features, and an Information Value screening pass.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

RAW_PATH = 'accepted_2007_to_2018Q4.csv'

## 1. Load and define the target

Same logic as the EDA notebook: only loans with a *realized* outcome can be labeled.

In [2]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(df.shape)

completed_statuses = [
    'Fully Paid', 'Charged Off', 'Default',
    'Late (31-120 days)', 'Late (16-30 days)',
]
df = df[df['loan_status'].isin(completed_statuses)].copy()

bad_map = {
    'Fully Paid': 0, 'Charged Off': 1, 'Default': 1,
    'Late (31-120 days)': 1, 'Late (16-30 days)': 1,
}
df['bad_flag'] = df['loan_status'].map(bad_map)
print(df.shape, ' | bad rate:', df['bad_flag'].mean().round(4))

(2260701, 151)
(1371166, 152)  | bad rate: 0.2147


## 2. Loan-maturity (seasoning) filter

As shown in the EDA notebook, restricting to "completed" loans quietly biases recent
vintages toward fast defaulters, since good loans haven't had time to reach `Fully
Paid` yet. We remove loans that haven't had enough time to mature given their term.

We use the latest `issue_d` in the file as a stand-in for "today." 

In [3]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['term_num'] = df['term'].str.extract(r'(\d+)').astype(int)

snapshot_date = df['issue_d'].max()
maturity_cutoff = {
    36: snapshot_date - pd.DateOffset(months=36),
    60: snapshot_date - pd.DateOffset(months=60),
}
print('Snapshot date proxy:', snapshot_date.date())
print('36-month loans must be issued on/before:', maturity_cutoff[36].date())
print('60-month loans must be issued on/before:', maturity_cutoff[60].date())

before = len(df)
df = df[
    ((df['term_num'] == 36) & (df['issue_d'] <= maturity_cutoff[36])) |
    ((df['term_num'] == 60) & (df['issue_d'] <= maturity_cutoff[60]))
].copy()
print(f"Dropped {before - len(df):,} immature loans -> {len(df):,} remain")
print('Bad rate after maturity filter:', df['bad_flag'].mean().round(4))

Snapshot date proxy: 2018-12-01
36-month loans must be issued on/before: 2015-12-01
60-month loans must be issued on/before: 2013-12-01
Dropped 697,544 immature loans -> 673,622 remain
Bad rate after maturity filter: 0.1482


Notice how much of the raw date range this removes — your *usable* modeling window is
smaller than your *raw data* window, purely because outcomes take years to reveal
themselves. This is a real constraint with any "time-to-event"
label (churn, default, warranty claims): the newest data is the least useful for
training, precisely because you don't yet know the truth about it.

## 3. Drop identifiers and no-signal columns

In [4]:
df = df.drop(columns=['id', 'member_id', 'title', 'emp_title', 'url', 'desc',
                       'funded_amnt', 'funded_amnt_inv'], errors='ignore')
# funded_amnt / funded_amnt_inv are near-duplicates of loan_amnt (EDA notebook, section 8)

## 4. Leakage screening

**The rule:** if a field can change, or only gets populated, *after* the loan is
funded, it cannot be used — the model would be trained on information that doesn't
exist yet at the moment a real application needs to be scored. Payment history,
recoveries, hardship/settlement fields, and "last/next payment date" columns are the
classic offenders in this dataset. A keyword screen catches most of them
automatically; a manual allow-list rescues the handful of legitimately safe columns
that happen to share a keyword (e.g. `pub_rec` contains "rec" but is a
pre-application bureau field, not a payment-recovery field).

In [5]:
leakage_keywords = [
    'pymnt', 'prncp', 'recover', 'rec', 'hardship', 'settlement', 'collection',
]
unsafe_date_keywords = ['last_', 'next_', '_d', 'date']
leakage_keywords_all = leakage_keywords + unsafe_date_keywords

# Columns that trip the keyword screen but are actually known pre-funding / safe to keep
safe_cols = [
    'earliest_cr_line', 'issue_d', 'pymnt_plan',
    'pub_rec', 'pub_rec_bankruptcies',
    'mths_since_last_record', 'mths_since_last_delinq', 'mths_since_last_major_derog',
    'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
]

leak_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in leakage_keywords_all) and c not in safe_cols
]
print(f"{len(leak_cols)} columns flagged as leakage:")
print(leak_cols)
df = df.drop(columns=leak_cols)
print(df.shape)

44 columns flagged as leakage:
['inq_last_6mths', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'acc_now_delinq', 'inq_last_12m', 'num_tl_90g_dpd_24m', 'pct_tl_nvr_dlq', 'sec_app_inq_last_6mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term']
(673622, 101)


## 5. Drop near-empty columns

Anything missing for the overwhelming majority of rows carries essentially no signal
and isn't worth the imputation risk (a column that's 95% imputed is basically a
constant with some noise).

In [6]:
missing_ratio = df.isna().mean()
near_empty = missing_ratio[missing_ratio > 0.70].index.tolist()
print(f"Dropping {len(near_empty)} columns >70% missing")
df = df.drop(columns=near_empty)
print(df.shape)

Dropping 31 columns >70% missing
(673622, 70)


## 6. Domain-aware missing value handling

Two buckets, as identified in EDA:
- **"Missing means zero/none"** — e.g. no public records, no bankruptcies filed. Filling
  with 0 (rather than the median) preserves the actual meaning.
- **"Missing means unknown"** — a bureau field just wasn't populated for that
  application. Median imputation is the safer default here; for a production system
  you'd typically also add a `<col>_was_missing` indicator so the model can use
  "unknown-ness" itself as a signal, which is standard practice when missingness rates
  are non-trivial (we add that below for the highest-missingness numeric columns).

In [7]:
zero_logical_cols = [
    'pub_rec', 'pub_rec_bankruptcies', 'delinq_2yrs', 'chargeoff_within_12_mths',
    'tax_liens', 'delinq_amnt', 'acc_now_delinq', 'collections_12_mths_ex_med',
]
zero_logical_cols = [c for c in zero_logical_cols if c in df.columns]
df[zero_logical_cols] = df[zero_logical_cols].fillna(0)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
median_fill_cols = [c for c in numeric_cols
                     if c not in zero_logical_cols and c != 'bad_flag'
                     and df[c].isna().any()]

# Missingness indicators for the columns with non-trivial missing rates,
# added *before* filling so the indicator reflects the true original gaps
for c in median_fill_cols:
    if df[c].isna().mean() > 0.02:
        df[f'{c}_was_missing'] = df[c].isna().astype(int)

for c in median_fill_cols:
    df[c] = df[c].fillna(df[c].median())

print('Remaining nulls (numeric):', df[numeric_cols].isna().sum().sum())

Remaining nulls (numeric): 0


In [8]:
# A few categorical columns can have missing values too (e.g. emp_length)
cat_cols_all = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols_all:
    if df[c].isna().any():
        df[c] = df[c].fillna('Unknown')

print('Remaining nulls (categorical):', df[cat_cols_all].isna().sum().sum())

Remaining nulls (categorical): 0


C:\Users\Admin\AppData\Local\Temp\ipykernel_22740\4090120492.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_all = df.select_dtypes(include='object').columns.tolist()


## 7. Feature engineering

A handful of ratio/derived features that are standard in credit scoring and aren't
already present as raw columns. All of these are computable strictly from information
known at application time.

In [9]:
df['credit_history_years'] = (
    (df['issue_d'] - pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')).dt.days / 365.25
)

if {'fico_range_low', 'fico_range_high'}.issubset(df.columns):
    df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
    df = df.drop(columns=['fico_range_low', 'fico_range_high'])

df['loan_to_income'] = df['loan_amnt'] / df['annual_inc'].clip(lower=1)
df['installment_to_income'] = (df['installment'] * 12) / df['annual_inc'].clip(lower=1)

if {'open_acc', 'total_acc'}.issubset(df.columns):
    df['open_acc_ratio'] = df['open_acc'] / df['total_acc'].clip(lower=1)

# sub_grade already encodes grade at finer granularity -> keep sub_grade, drop grade
# (same reasoning already applied in the existing modeling.ipynb)
if 'grade' in df.columns:
    df = df.drop(columns=['grade'])

df = df.drop(columns=['earliest_cr_line'])  # fully captured by credit_history_years now
print(df.shape)
df[['credit_history_years', 'loan_to_income', 'installment_to_income']].describe()

(673622, 105)


,credit_history_years,loan_to_income,installment_to_income
count,673622.000000,673622.000000,673622.000000
mean,16.091306,0.238389,0.093893
std,7.525946,24.781548,10.580401
min,2.997947,0.000309,0.000123
25%,10.997947,0.118750,0.046025
50%,14.584531,0.187500,0.072465
75%,19.832991,0.275568,0.106269
max,71.000684,20000.000000,8555.520000


## 8. Information Value (IV) screening — a leakage sanity-check, not just a metric

Weight of Evidence (WOE) and Information Value (IV) are credit-risk-specific tools
that show up constantly in this domain (bank scorecards, regulatory model
documentation) even when the final model is a gradient-boosted tree that never
touches WOE directly. WOE for a bin is `ln(% of goods in bin / % of bads in bin)`; IV
sums a weighted version of that across all bins of a feature into a single number.

**Rule-of-thumb IV bands** :
| IV | Interpretation |
|---|---|
| < 0.02 | Not predictive |
| 0.02 - 0.1 | Weak |
| 0.1 - 0.3 | Medium |
| 0.3 - 0.5 | Strong |
| > 0.5 | Suspiciously strong — investigate for leakage |

That last row is the real reason to run this now, right after the leakage screen: if a
feature you *think* is clean scores IV > 0.5, that's a prompt to go check it, not a
reason to celebrate a great feature.

In [10]:
def compute_iv(frame, feature, target, bins=10):
    x = frame[feature]
    if x.dtype.kind in 'ifu' and x.nunique() > bins:
        binned = pd.qcut(x, q=bins, duplicates='drop')
    else:
        binned = x.astype(str)
    tmp = pd.DataFrame({'bin': binned, 'y': frame[target].values})
    g = tmp.groupby('bin', observed=True)['y'].agg(total='count', bad='sum')
    g['good'] = g['total'] - g['bad']
    total_bad, total_good = g['bad'].sum(), g['good'].sum()
    # Laplace smoothing avoids -inf/inf WOE for empty bins
    g['dist_bad'] = (g['bad'] + 0.5) / (total_bad + 0.5 * len(g))
    g['dist_good'] = (g['good'] + 0.5) / (total_good + 0.5 * len(g))
    g['woe'] = np.log(g['dist_good'] / g['dist_bad'])
    g['iv'] = (g['dist_good'] - g['dist_bad']) * g['woe']
    return g, g['iv'].sum()

candidate_features = [c for c in df.columns if c not in
                      ['bad_flag', 'loan_status', 'issue_d', 'zip_code']]

iv_summary = []
for feat in candidate_features:
    try:
        _, iv = compute_iv(df, feat, 'bad_flag')
        iv_summary.append((feat, iv))
    except Exception:
        pass

iv_table = pd.DataFrame(iv_summary, columns=['feature', 'iv']).sort_values(
    'iv', ascending=False).reset_index(drop=True)
iv_table['band'] = pd.cut(
    iv_table['iv'], bins=[-np.inf, 0.02, 0.1, 0.3, 0.5, np.inf],
    labels=['not predictive', 'weak', 'medium', 'strong', 'CHECK ME']
)
iv_table.head(20)

,feature,iv,band
0,sub_grade,0.387367,strong
1,int_rate,0.350064,strong
2,fico_avg,0.125435,medium
3,acc_open_past_24mths,0.068121,weak
4,installment_to_income,0.065663,weak
5,bc_open_to_buy,0.065249,weak
6,loan_to_income,0.058309,weak
7,total_bc_limit,0.055487,weak
8,annual_inc,0.055140,weak
9,num_tl_op_past_12m,0.052596,weak


In [11]:
suspicious = iv_table[iv_table['band'] == 'CHECK ME']
print(f"{len(suspicious)} feature(s) flagged for manual review:")
suspicious

0 feature(s) flagged for manual review:


,feature,iv,band


If anything shows up here, need to look at it before modeling — either it's a genuinely
strong, legitimate signal (rare but possible for something like `sub_grade`/`int_rate`,
which really are that predictive because they're themselves risk assessments), or it's
a leak that slipped past the keyword screen. We don't drop features solely because they
have high IV — confirm *why* first.

## 9. Out-of-time (OOT) train / validation / test split

We split by `issue_d`, not randomly. Training on the past and validating on the future
is the only split design that actually rehearses deployment: a model trained today
will only ever score applications that come in *after* training, never applications
mixed in from the same time window. A random split hides any performance degradation
caused by population drift — because the OOT split is *designed* to expose exactly that,
if it's there.

In [12]:
df = df.sort_values('issue_d').reset_index(drop=True)

train_cut = df['issue_d'].quantile(0.70)
val_cut = df['issue_d'].quantile(0.85)

train_df = df[df['issue_d'] <= train_cut].copy()
val_df = df[(df['issue_d'] > train_cut) & (df['issue_d'] <= val_cut)].copy()
test_df = df[df['issue_d'] > val_cut].copy()

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name:<6} n={len(part):>8,}  "
          f"range=[{part['issue_d'].min().date()} -> {part['issue_d'].max().date()}]  "
          f"bad_rate={part['bad_flag'].mean():.4f}")

train  n= 492,256  range=[2007-06-01 -> 2015-05-01]  bad_rate=0.1483
val    n=  92,633  range=[2015-06-01 -> 2015-09-01]  bad_rate=0.1474
test   n=  88,733  range=[2015-10-01 -> 2015-12-01]  bad_rate=0.1488


If the bad rate differs noticeably between train and test here, that's not a bug to
fix — it's the population drift the whole OOT design exists to surface. A model that
looks great in-sample but wobbles on the OOT test set is telling you something a random
split never would have.

**Why not `TimeSeriesSplit`/k-fold for the final split?** Time-series cross-validation
(expanding-window folds) is genuinely useful *during hyperparameter tuning* — we use it
that way in notebook 03. For the final held-out evaluation, though, a single clean
train/val/test cut is easier to reason about and matches how the model will actually be
validated before a real deployment sign-off: one fixed development window, one fixed
future-period test.

In [13]:
drop_for_modeling = ['loan_status', 'issue_d', 'zip_code']
# zip_code (masked, ~900 distinct 3-digit prefixes) is dropped for the reference
# pipeline to avoid a very high-cardinality, weakly-informative geographic proxy;
# addr_state captures state-level geography at a modeling-friendly cardinality.
# (Keep this decision in mind for the fairness discussion in notebook 06 — geography
# fields are also the most common proxy-discrimination risk in credit models.)

train_df = train_df.drop(columns=drop_for_modeling, errors='ignore')
val_df = val_df.drop(columns=drop_for_modeling, errors='ignore')
test_df = test_df.drop(columns=drop_for_modeling, errors='ignore')

train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)
test_df.to_csv('test.csv', index=False)
iv_table.to_csv('iv_table.csv', index=False)

print('Saved train.csv, val.csv, test.csv, iv_table.csv')
print('Final feature count (excluding target):', train_df.shape[1] - 1)

Saved train.csv, val.csv, test.csv, iv_table.csv
Final feature count (excluding target): 101


## Summary

| Stage | Rows removed | Why |
|---|---|---|
| Completed-status filter | Open/in-progress loans | Outcome not yet known |
| Maturity filter | Recent, not-yet-seasoned loans | Survivorship bias toward fast defaults |

| Decision | What we did | Why |
|---|---|---|
| Encoding | Deferred to the modeling `Pipeline` | Fit only on train folds -> no leakage, and it travels with the model artifact |
| High-missingness columns | Dropped (>70% missing) | Little signal left after that much imputation |
| Structural-zero columns | Filled with 0 | Missing means "none", not "unknown" |
| Other numeric columns | Median-filled + missingness indicator | Missing means "unknown"; indicator preserves that signal |
| `grade` | Dropped, kept `sub_grade` | Redundant — sub_grade is a strict refinement |
| Split | Out-of-time (70/15/15 by `issue_d`) | Simulates real deployment; a random split would hide temporal drift |

Next: **03 — imbalance handling and modeling.**